 Live survey response analysis

In [ ]:
import json
from urllib.request import Request, urlopen

import pandas as pd
from IPython.display import Markdown, display

from survey_responce import (
	extract_triplets_by_category,
	limit_response_cases,
	rank_borda,
	rank_by_category,
	rank_overall,
	rank_plackett_luce,
	summarize_participants,
)


In [ ]:
config_path = "live-analisys-config.json"
survey_path = "../eval/output/survey.json"

with open(config_path, encoding="utf-8") as config_file:
	config = json.load(config_file)


def fetch_api(path):
	request = Request(
		f"{config['base_url'].rstrip('/')}{path}",
		headers={"Authorization": f"Bearer {config['admin_token']}"},
	)
	with urlopen(request, timeout=30) as api_response:
		return json.load(api_response)


n_cases = 99
responses = fetch_api("/api/responses")
response_labels = pd.Series([response.get("label") for response in responses], dtype="string")
responses = [
	response
	for response, include in zip(responses, response_labels.str.match(r"^p\d{2}", na=False))
	if include
]
responses = limit_response_cases(responses, survey_path, n_cases)
participants = fetch_api("/api/participants/")


 Responses by participant

In [ ]:
participant_stats = pd.DataFrame(
	summarize_participants(responses, participants, survey_path),
	columns=[
		"participant", "group", "status", "last_page", "saved_at",
		"completed_cases", "incomplete_cases", "complete_rankings",
	],
)
participant_stats["group"] = pd.to_numeric(participant_stats["group"]).astype("Int64")
participant_stats = participant_stats.astype({"last_page": "Int64"})
status_counts = participant_stats["status"].value_counts()
display(Markdown(
	f"**{len(participant_stats)}** participants with responses: "
	f"**{status_counts.get('submitted', 0)}** submitted, **{status_counts.get('draft', 0)}** drafts; "
	f"**{participant_stats['completed_cases'].sum()}** completed cases and "
	f"**{participant_stats['complete_rankings'].sum()}** complete rankings in total."
))
participant_stats.sort_values(["status", "participant"], ascending=[False, True], ignore_index=True)


Rankings by category

In [ ]:
# Drafts are autosaved on every change, so rows without both best and worst are skipped.
triplets_by_category = extract_triplets_by_category(responses, survey_path, skip_incomplete=True)
rank_method = rank_borda
# rank_method = rank_plackett_luce 

for category, rankings in rank_by_category(triplets_by_category, rank_method).items():
	display(Markdown(f"### {category}"))
	display(pd.DataFrame(rankings).set_index("rank"))

if triplets_by_category:
	display(Markdown("### Overall"))
	display(pd.DataFrame(rank_overall(triplets_by_category, rank_method)).set_index("rank"))
else:
	display(Markdown("No complete rankings yet."))


### Participant question timeline

First recorded appearance of each question in saved answers (UTC), not completion time. This timeline includes all recorded questions, independently of `n_cases`.

In [ ]:
timeline_participant = "p09"
timeline_matches = [participant for participant in participants if participant["label"] == timeline_participant]
if len(timeline_matches) != 1:
	raise ValueError(f"Expected one participant labelled {timeline_participant!r}; found {len(timeline_matches)}.")

participant_questions = fetch_api("/api/participant-questions")
question_timeline = pd.DataFrame(
	[
		{"question_name": question["question_name"], "first_answered_at": question["first_answered_at"]}
		for question in participant_questions
		if question["token"] == timeline_matches[0]["token"]
	],
	columns=["question_name", "first_answered_at"],
)
question_timeline["first_answered_at"] = pd.to_datetime(question_timeline["first_answered_at"], format="ISO8601", utc=True, errors="raise")
question_timeline = question_timeline.sort_values("first_answered_at", kind="stable", ignore_index=True)
display(Markdown(f"### Question timeline: {timeline_participant}"))
if question_timeline.empty:
	display(Markdown("No question timestamps recorded for this participant."))
else:
	with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
		display(question_timeline.assign(first_answered_at=question_timeline["first_answered_at"].dt.strftime("%Y-%m-%d %H:%M")))
